# Lab 59 (solution): Re-tuning the threshold on held-out pairs

Reference implementation. When you swap in a real sentence-transformer ([Lab 55](../../55-calibrated-detection-judgment/) `embedders.py`) and re-tune the change-detection threshold, you tune *out-of-sample*: in-sample selection is optimistically biased, held-out cross-validation is the fix, and the optimism curve tells you how many pairs to label. Math: [math-foundations/16](../../../math-foundations/16-threshold-selection-under-shift.md).

## Step 0: Setup

In [ ]:
from retune import make_pairs, optimism_curve, tune_threshold, cv_accuracy, accuracy_at
# Lab 55 tuned the threshold; this lab re-tunes OUT-OF-SAMPLE. In-sample tuning (tune and report on the
# same pairs) is optimistically biased; held-out cross-validation is the fix. How big the bias is
# depends on how many labeled pairs you have.
pairs = make_pairs()
print(f"{len(pairs)} labeled reflow/edit pairs (with hard cases, so the classes overlap)")

## Step 1: The optimism of in-sample tuning, by sample size

In [ ]:
# The optimism of in-sample threshold selection, as a function of sample size:
curve = optimism_curve(pairs)
print(f"{'n':>5} {'in-sample':>10} {'held-out':>10} {'optimism':>10}")
for n,r in curve.items():
    print(f"{n:>5} {r['in_sample']:>10.3f} {r['held_out']:>10.3f} {r['optimism']:>+10.3f}")
print("\nIn-sample accuracy is inflated at small n and the gap vanishes as n grows - which is how")
print("you decide how many reflow/edit pairs to label before trusting a re-tuned threshold.")

## Step 2: Re-tuning with a real embedder

In [ ]:
# Swapping in a real sentence-transformer is a one-line change; the procedure is identical.
# (The dependency loads only when you instantiate it; offline, the char-trigram stand-in is used.)
print("from embedders import SentenceTransformerEmbedder, tune_with")
print("tune_with(SentenceTransformerEmbedder(), held_out_pairs)   # re-tune on real embeddings")
print("\nThen run the SAME optimism_curve on the real-embedding cosines to size your label budget.")

## What you built

An unbiased re-tuning workflow. `retune.py` builds overlapping reflow/edit pairs (with hard cases - paraphrases that drop the cosine, negations that barely move it), then measures the **optimism** of in-sample threshold selection: tuning the threshold and reporting accuracy on the same pairs inflates the number, and the inflation shrinks as the sample grows (here ~+0.03 at n=16, ~0 by n=160). The fix is cross-validated held-out evaluation, and the curve tells you how many labeled pairs you need before a re-tuned threshold is trustworthy.

**Where this simplifies:** the embedder is the deterministic char-trigram stand-in so the lab runs offline; `--real-embedder` uses a sentence-transformer when installed, and the workflow - hold out, cross-validate, watch the optimism curve - is identical. The math is in [math-foundations/16](../../../math-foundations/16-threshold-selection-under-shift.md). The deeper point: a threshold is a fitted parameter, so it earns the same train/validation discipline as any model - re-tune it when the embedder or the data distribution changes, and validate it out of sample.